# CDSE S3 Bucket Download Tutorial (STAC API)

This tutorial demonstrates how to:
1. Search for Sentinel-1 SLC (`IW` mode, `VV`/`VH` pol) data over Vienna using the CDSE STAC API.
2. Resolve the S3 path for the product using the CDSE OData API.
3. Download the full 4-8 GB product via the CDSE S3 interface using Boto3.

### Prerequisites
Make sure your CDSE S3 keys are loaded as environment variables:
- `cdse_S3_KEY`
- `cdse_S3_SECRET`

In [ ]:
import os
import time
import requests
import boto3
import concurrent.futures

# Load S3 credentials from environment variables
ACCESS_KEY = os.environ.get('cdse_S3_KEY')
SECRET_KEY = os.environ.get('cdse_S3_SECRET')

DOWNLOAD_PATH = f"{os.getenv('HOME')}/bucket/demo_data/"

if not ACCESS_KEY or not SECRET_KEY:
    print("Warning: S3 credentials not found. Please set them before running the download step.")

## 1. Search for a Scene via CDSE STAC API
We query the `sentinel-1-slc` collection using a bounding box and date range, then filter the results for our specific Beam Mode (`IW`) and Polarization (`VV`).

In [ ]:
stac_endpoint = "https://stac.dataspace.copernicus.eu/v1/search"

# Bounding box for Vienna [min_lon, min_lat, max_lon, max_lat]
vienna_bbox = [16.20, 48.10, 16.60, 48.30]

search_payload = {
    "collections": ["sentinel-1-slc"],
    "bbox": vienna_bbox,
    "datetime": "2024-05-01T00:00:00Z/2024-05-10T00:00:00Z",
    "limit": 20
}

print("Searching CDSE STAC API...")
response = requests.post(stac_endpoint, json=search_payload)
response.raise_for_status()

items = response.json().get('features', [])
if not items:
    raise ValueError("No scenes found matching the spatial and temporal criteria.")

# Filter the results to match our specific parameters (Beam Mode = IW, Polarization contains VV)
selected_scene = None
for item in items:
    props = item.get('properties', {})
    
    instrument_mode = props.get('sar:instrument_mode', '')
    polarizations = props.get('sar:polarizations', [])
    
    if instrument_mode == 'IW' and 'VV' in polarizations:
        selected_scene = item
        break

if not selected_scene:
    raise ValueError("Scenes were found, but none matched the IW mode and VV polarization criteria.")

# STAC 'id' for Sentinel-1 drops the .SAFE extension, but OData needs it.
base_id = selected_scene['id']
product_name = base_id if base_id.endswith('.SAFE') else f"{base_id}.SAFE"
acquisition_date = selected_scene['properties'].get('datetime')

print(f"Found Matching Scene: {product_name}")
print(f"Date: {acquisition_date}")
print(f"Mode: {selected_scene['properties'].get('sar:instrument_mode')}")
print(f"Polarizations: {selected_scene['properties'].get('sar:polarizations')}")

## 2. Get the S3 Path via CDSE OData API
We use the Copernicus Data Space Ecosystem OData API to pinpoint the exact internal S3 path for the `.SAFE` product.

In [ ]:
odata_url = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Name eq '{product_name}'"
odata_response = requests.get(odata_url)
odata_response.raise_for_status()

product_data = odata_response.json().get('value', [])
if not product_data:
    raise ValueError(f"Product {product_name} not found in CDSE OData catalogue.")

s3_path = product_data[0]['S3Path']
print(f"OData S3Path: {s3_path}")

# Parse bucket and prefix from the path (e.g., /eodata/Sentinel-1/... )
path_parts = s3_path.lstrip('/').split('/', 1)
bucket_name = path_parts[0]
base_s3_prefix = path_parts[1]
print(f"Bucket: {bucket_name}, Prefix: {base_s3_prefix}")

## 3. Download from CDSE S3
Finally, we connect to `eodata.dataspace.copernicus.eu` to retrieve the files. 
*Note: Sentinel-1 SLC datasets are massive (often 4-8GB), so this step will take some time especially if you are running it locally it will depend on your connection.*

In [ ]:
def download_single_file(s3_key, bucket_name, local_file_path):
    """
    Helper function to download a single file. 
    Creates a new Boto3 session to ensure thread safety.
    """
    # Skip if it's just a directory marker
    if local_file_path.endswith('/'):
        os.makedirs(local_file_path, exist_ok=True)
        return
        
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
    
    # Boto3 sessions are NOT thread-safe, so we instantiate a new client per thread
    session = boto3.session.Session()
    s3_client = session.client(
        's3',
        endpoint_url='https://eodata.dataspace.copernicus.eu',
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        region_name='default'
    )
    
    s3_client.download_file(bucket_name, s3_key, local_file_path)

def download_product_from_s3_parallel(bucket_name, prefix, target_dir, max_threads=8):
    # Initial resource just to list the objects
    s3_resource = boto3.resource(
        's3',
        endpoint_url='https://eodata.dataspace.copernicus.eu',
        aws_access_key_id=ACCESS_KEY,
        aws_secret_access_key=SECRET_KEY,
        region_name='default'
    )
    
    bucket = s3_resource.Bucket(bucket_name)
    objects = list(bucket.objects.filter(Prefix=prefix))
    
    if not objects:
        print("No files found for this product prefix in CDSE S3.")
        return
    
    # Calculate total size before downloading
    total_bytes = sum(obj.size for obj in objects)
    total_mb = total_bytes / (1024 * 1024)
    
    print(f"Starting parallel download of {len(objects)} files (Total size: {total_mb:.2f} MB) to '{target_dir}'...")
    print(f"Using up to {max_threads} concurrent threads.")
    
    start_time = time.time()
    
    # Use ThreadPoolExecutor for parallel downloads
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = []
        for obj in objects:
            relative_path = os.path.relpath(obj.key, prefix)
            local_file_path = os.path.join(target_dir, relative_path)
            
            # Submit task to the thread pool
            futures.append(
                executor.submit(download_single_file, obj.key, bucket_name, local_file_path)
            )
            
        # Wait for all futures to complete
        concurrent.futures.wait(futures)
            
    end_time = time.time()
    
    duration_sec = end_time - start_time
    speed_mbps = total_mb / duration_sec if duration_sec > 0 else 0
    
    print(f"\nDownload complete! Files saved in: {os.path.abspath(target_dir)}")
    print("\n--- Parallel Download Statistics ---")
    print(f"Total Data Downloaded: {total_mb:.2f} MB")
    print(f"Total Time Taken:      {duration_sec:.2f} seconds")
    print(f"Average Speed:         {speed_mbps:.2f} MB/s")

# Execute the download
target_location = f"{DOWNLOAD_PATH}{product_name}"
if ACCESS_KEY and SECRET_KEY:
    download_product_from_s3_parallel(bucket_name, base_s3_prefix, target_dir=target_location, max_threads=8)
else:
    print("Skipping download. Missing S3 credentials.")